<a href="https://colab.research.google.com/github/naurasafina/python-google-colab-projek/blob/main/simulasi_antrian_momoyo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta, datetime

In [ ]:
# Menampilkan semua baris data
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
# menghitung semua metrik waktu yang terkait dengan proses layanan satu pelanggan dalam skenario antrian Momoyo
def calculate_time_metrics(arrival_time, last_service_end_time, service_time_dist):

# menghitung waktu mulai dilayani
    start_service_time = max(arrival_time, last_service_end_time)
    min_svc, max_svc = service_time_dist

# menghitung durasi pelayanan
    service_duration_minutes = random.uniform(min_svc, max_svc)

# menghitung waktu selesai pelayanan
    service_end_time = start_service_time + timedelta(minutes=service_duration_minutes)

# Menghitung waktu tuggu
    wait_time_minutes = (start_service_time - arrival_time).total_seconds() / 60

    return start_service_time, service_end_time, round(wait_time_minutes, 2), round(service_duration_minutes, 2)

In [ ]:
# Fungsi Simulasi
def simulate_momoyo_queue_day(start_time_dt, end_time_dt, day_name, service_time_dist):

    data = []
    current_arrival_time = start_time_dt
    last_service_end_time = start_time_dt
    customer_id_counter = 1

# Mengontrol jam oprasional
    while current_arrival_time < end_time_dt:
        current_hour = current_arrival_time.hour

        # Modeling untuk jam sibuk
        if 12 <= current_hour < 14:  # Jam sibuk Makan Siang
            mean_interval = 10.0   # Rata-rata 1 pelanggan setiap 10.0 menit
        elif 16 <= current_hour < 19: # Jam sibuk Sore
            mean_interval = 12.0   # Rata-rata 1 pelanggan setiap 12.0 menit
        else:                        # Jam Normal
            mean_interval = 14.0   # Rata-rata 1 pelanggan setiap 14.0 menit

        # Waktu datang antar pelanggan
        inter_arrival_time_minutes = np.random.exponential(scale=mean_interval)
        arrival_time = current_arrival_time + timedelta(minutes=inter_arrival_time_minutes)

        # Pemeriksaan jam tutup
        if arrival_time > end_time_dt:
            break

        # Logika antrian
        start_svc, end_svc, wait_time, svc_duration = calculate_time_metrics(
            arrival_time, last_service_end_time, service_time_dist
        )

        # Hasil dari semua matriks yang sudah dihitung
        data.append({
            'Hari': day_name,
            'No': f'C{customer_id_counter}',
            'Waktu Kedatangan': arrival_time.strftime('%H:%M'),
            'Waktu Mulai Dilayani': start_svc.strftime('%H:%M'),
            'Waktu Selesai Pelayanan': end_svc.strftime('%H:%M'),
            'Waktu Tunggu (menit)': wait_time
        })

        current_arrival_time = arrival_time
        last_service_end_time = end_svc
        customer_id_counter += 1

    return data

In [ ]:
# Fungsi Kontrol
def run_multi_day_simulation(days_to_simulate, start_hour, end_hour, svc_time_range):

    all_data = []

    # Kalender simulasi
    dates = {
        'Senin': datetime(2025, 11, 17),
        'Selasa': datetime(2025, 11, 18),
        'Rabu': datetime(2025, 11, 19)
    }

    # Hari kerja
    for day in days_to_simulate:
        if day not in dates:
            continue

        # Waktu buka dan tutup
        date = dates[day]
        start_dt = date.replace(hour=start_hour, minute=0, second=0)
        end_dt = date.replace(hour=end_hour, minute=0, second=0)

        day_data = simulate_momoyo_queue_day(start_dt, end_dt, day, svc_time_range)
        all_data.extend(day_data)

    # Konversi menjadi tabel
    df_simulasi = pd.DataFrame(all_data)

    return df_simulasi

In [ ]:
# Parameter Simulasi

# Hari yang disimulasikan
DAYS = ['Senin', 'Selasa', 'Rabu']

# Jam oprasi
START = 9  # Jam buka 09:00
END = 21   # Jam tutup 21:00

# Asumsi layanan
SERVICE_TIME_RANGE = (2, 5) # Waktu pelayanan (min 2, max 5 menit)

# Menjalankan simulasi
df_final = run_multi_day_simulation(DAYS, START, END, SERVICE_TIME_RANGE)

# Menampilkan hasil simulasi
print("HASIL SIMULASI ANTRIAN MOMOYO")
print(f"Total Pelanggan Terlayani Selama 3 Hari: {len(df_final)}\n")

# Menggunakan .to_string() untuk menampilkan SELURUH data tanpa pemotongan
print(df_final.to_string())

print("\nRingkasan Statistik")
print(f"Rata-rata Waktu Tunggu Seluruh Pelanggan: {df_final['Waktu Tunggu (menit)'].mean():.2f} menit")
print(f"Waktu Tunggu Maksimum yang Tercatat: {df_final['Waktu Tunggu (menit)'].max():.2f} menit")

HASIL SIMULASI ANTRIAN MOMOYO
Total Pelanggan Terlayani Selama 3 Hari: 155

       Hari   No Waktu Kedatangan Waktu Mulai Dilayani Waktu Selesai Pelayanan  Waktu Tunggu (menit)
0     Senin   C1            09:00                09:00                   09:04                  0.00
1     Senin   C2            09:01                09:04                   09:08                  2.90
2     Senin   C3            09:06                09:08                   09:12                  2.28
3     Senin   C4            09:25                09:25                   09:28                  0.00
4     Senin   C5            10:39                10:39                   10:44                  0.00
5     Senin   C6            11:06                11:06                   11:10                  0.00
6     Senin   C7            12:18                12:18                   12:20                  0.00
7     Senin   C8            12:18                12:20                   12:22                  1.91
8     Senin   C